In [34]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.optim as optim
import numpy as np
import pandas as pd 
import json

from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score
from medmnist import INFO

from tqdm.notebook import tqdm

from medmnist import ChestMNIST

In [2]:
import ctypes

cuda = ctypes.CDLL("/run/opengl-driver/lib/libcuda.so.1")
print("CUDA initialization:", cuda.cuInit(0))


CUDA initialization: 0


In [3]:
torch.cuda.get_device_name()

'NVIDIA GeForce RTX 3050 Laptop GPU'

In [4]:
device = torch.device("cuda")
device

device(type='cuda')

In [5]:
Loss = nn.BCEWithLogitsLoss() #over crossEntropy BCE lets each label work with it's own signmoid

In [6]:
trans = transforms.Compose([transforms.ToTensor(),
                            transforms.Normalize(mean=[0.5], std=[0.5])])

In [7]:
train_df = ChestMNIST(split="train", transform=trans ,download=True)
val_df = ChestMNIST(split="val", transform=trans, download=True)
test_df = ChestMNIST(split="test", transform=trans, download=True)

In [8]:
batch_size = 64 

train_loader = DataLoader(train_df, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_df, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_df, batch_size=batch_size, shuffle=False)

In [9]:
images, labels = next(iter(train_loader))

f"{images.shape}, {labels.shape}, {images.dtype}, {labels.dtype}"

'torch.Size([64, 1, 28, 28]), torch.Size([64, 14]), torch.float32, torch.int64'

In [10]:
from torchvision.models import resnet18
model = resnet18(weights=None)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [11]:
model.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(in_features=model.fc.in_features, out_features=14)

In [12]:
model = model.to(device)

In [13]:
images, labels = next(iter(train_loader))

images = images.to(device)

In [14]:
with torch.no_grad():
    op = model(images)

f"{images.shape}, {op.shape}"

'torch.Size([64, 1, 28, 28]), torch.Size([64, 14])'

In [15]:
optim = optim.Adam(model.parameters(), lr=0.01)

In [16]:
def epoch0(model, train_loader, Loss, optim, device):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc="Training", leave=False):

        images = images.to(device)
        labels = labels.float().to(device)

        optim.zero_grad()

        ops = model(images)

        loss = Loss(ops, labels)

        loss.backward()

        optim.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(train_loader.dataset)

In [17]:
@torch.no_grad()
def val0(model, val_loader, Loss, device):
    model.eval()
    running_loss = 0.0

    for images, labels in tqdm(val_loader, desc="Validating", leave=False):

        images = images.to(device)
        labels = labels.float().to(device)

        ops = model(images)

        loss = Loss(ops, labels)

        running_loss += loss.item() * images.size(0)
    
    return running_loss / len(val_loader.dataset)

In [18]:
epochs = 10 

train_losses = []
val_losses = []

for epoch in range(epochs):

    train_loss = epoch0(model, train_loader, Loss, optim, device)
    val_loss = val0(model, val_loader, Loss, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch: {epoch+1}, Train loss: {train_loss}, val loss: {val_loss}")

Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 1, Train loss: 0.18054735836249497, val loss: 0.17537267913749505


Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 9, Train loss: 0.1630596732998463, val loss: 0.16392575556745886


Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 10, Train loss: 0.16199429267352283, val loss: 0.16669664465469392


In [20]:
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing rn"):
        images = images.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

y_prob = np.concatenate(all_probs, axis=0)
y_true = np.concatenate(all_labels, axis=0)

f"Prob shape: {y_prob.shape}, Labels shape: {y_true.shape}"

Testing rn:   0%|          | 0/351 [00:00<?, ?it/s]

'Prob shape: (22433, 14), Labels shape: (22433, 14)'

In [25]:
auc_score = []

for i in range(14):
    auc = roc_auc_score(y_true[:,i], y_prob[:,i])
    auc_score.append(auc)

f"macro auc score {np.mean(auc_score)}"

'macro auc score 0.7278914839379734'

In [26]:
y_pred = (y_prob >= 0.5).astype(int)

In [27]:
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

f"precision: {precision}, recall: {recall}, f1: {f1}"

'precision: 0.23386522798270365, recall: 0.026763447190321355, f1: 0.03553769482795186'

In [29]:
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

In [30]:
rows = []

for threshold in thresholds:
    preds = (y_prob >= threshold).astype(int)
    rows.append({
        "Threshold": threshold, "Precision": precision_score(y_true, preds, average="macro", zero_division=0),
        "Recall": recall_score(y_true, preds, average="macro", zero_division=0),
        "F1": f1_score(y_true, preds, average="macro", zero_division=0)})

results = pd.DataFrame(rows)
results

,Threshold,Precision,Recall,F1
0,0.05,0.098787,0.535924,0.154079
1,0.10,0.118332,0.333366,0.162000
2,0.15,0.178342,0.224060,0.141644
3,0.20,0.132299,0.162768,0.124772
4,0.25,0.150962,0.123764,0.109878
5,0.30,0.179971,0.095068,0.095223
6,0.35,0.178467,0.072772,0.080394
7,0.40,0.230830,0.053696,0.065740
8,0.45,0.170057,0.037489,0.048523
9,0.50,0.233865,0.026763,0.035538


In [31]:
torch.save(model.state_dict(), "resnet18_chestMNIST.pth")

In [33]:
model.eval()
dummy = torch.randn(1, 1, 28, 28).to(device)

torch.onnx.export(  model, (dummy,), "resnet18_chestMNIST.onnx", input_names=["input"], output_names=["logits"],
                    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},opset_version=17, dynamo=False   )


/tmp/ipykernel_194061/2845617342.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(  model, (dummy,), "resnet18_chestMNIST.onnx", input_names=["input"], output_names=["logits"],
